# AdS4 Jacobian / Conditioning Sweep (v12)

This notebook extends v10/v11 by replacing the naive overlap proxy with a local sensitivity analysis.

We:
- define probes EE, WL, GEO as functions of the latent geometry parameters f and g
- compute the local Jacobian of probes with respect to (f, g)
- extract conditioning diagnostics (singular values, condition number, determinant magnitude)
- compare conditioning collapse against the empirical corruption threshold c*

Goal:
- identify whether the loss of identifiability corresponds to a Jacobian ill-conditioning threshold
- move from overlap heuristics to a sharper inverse-problem metric


In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


In [ ]:
# Ground-truth AdS4-style toy background
r = torch.linspace(1.05, 3.5, 240, device=device).view(-1, 1)

def f_true(r):
    return r**2 + 1 - 1/r

def g_true(r):
    return 1/(f_true(r) + 1e-6)

def ee_from_fg(f, g):
    return torch.log(1.0 + 0.6 * f)

def wl_from_fg(f, g):
    return torch.sqrt(f + 1.8 * g + 1e-6)

def geo_from_fg(f, g):
    return torch.sqrt(f + 0.7 * g + 1e-6)

ee_obs = ee_from_fg(f_true(r), g_true(r)).detach()
wl_obs = wl_from_fg(f_true(r), g_true(r)).detach()
geo_obs = geo_from_fg(f_true(r), g_true(r)).detach()


In [ ]:
def structured_geo_target(r, corruption_strength=0.0):
    base = geo_from_fg(f_true(r), g_true(r))
    if corruption_strength == 0.0:
        return base.detach()
    radial_bias = 1.0 + corruption_strength * (0.30 * (r - r.min()) / (r.max() - r.min()))
    oscillation = 1.0 + corruption_strength * 0.15 * torch.sin(2.2 * r)
    return (base * radial_bias * oscillation).detach()


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = MLP()
        self.g = MLP()
    def forward(self, r):
        f = F.softplus(self.f(r))
        g = F.softplus(self.g(r))
        return f, g


In [ ]:
def train(corruption_strength=0.0, epochs=1200, consistency_weight=0.15):
    m = Model().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=2e-3)
    geo_target = structured_geo_target(r, corruption_strength)

    loss_hist = []
    consistency_hist = []

    for _ in range(epochs):
        opt.zero_grad()
        f, g = m(r)

        ee_pred = ee_from_fg(f, g)
        wl_pred = wl_from_fg(f, g)
        geo_pred = geo_from_fg(f, g)

        loss_ee = ((ee_pred - ee_obs)**2).mean()
        loss_wl = ((wl_pred - wl_obs)**2).mean()
        loss_geo = ((geo_pred - geo_target)**2).mean()
        loss_consistency = ((f * g - 1.0)**2).mean()

        loss = loss_ee + loss_wl + loss_geo + consistency_weight * loss_consistency
        loss.backward()
        opt.step()

        loss_hist.append(loss.item())
        consistency_hist.append(loss_consistency.item())

    with torch.no_grad():
        f_pred, g_pred = m(r)
        metric_err = ((f_pred - f_true(r)).abs() + (g_pred - g_true(r)).abs()) / 2.0
        abs_err_mean = metric_err.mean().item()
        abs_err_max = metric_err.max().item()
        consistency_final = consistency_hist[-1]

    return {
        'loss_hist': loss_hist,
        'consistency_hist': consistency_hist,
        'f_pred': f_pred,
        'g_pred': g_pred,
        'abs_err_mean': abs_err_mean,
        'abs_err_max': abs_err_max,
        'consistency_final': consistency_final,
    }


In [ ]:
# Reuse the v10-style sweep so we can compare conditioning to c*
strengths = [round(x, 2) for x in np.arange(0.00, 0.301, 0.02)]
results = []

for s in strengths:
    out = train(corruption_strength=float(s), epochs=1200)
    results.append({
        'strength': float(s),
        'abs_err_mean': out['abs_err_mean'],
        'abs_err_max': out['abs_err_max'],
        'consistency_final': out['consistency_final'],
        'loss_hist': out['loss_hist'],
        'f_pred': out['f_pred'],
        'g_pred': out['g_pred'],
    })
    print(f"corruption={s:.2f} | mean err={out['abs_err_mean']:.6f} | max err={out['abs_err_max']:.6f}")


In [ ]:
# Numerical threshold from mean error tolerance
mean_tolerance = 0.10
c_star_num = None
for row in results:
    if c_star_num is None and row['abs_err_mean'] > mean_tolerance:
        c_star_num = row['strength']
        break
print('numeric c*:', c_star_num)


In [ ]:
# Local Jacobian / conditioning analysis at the learned solution for each corruption level
def jacobian_condition_metrics(f0, g0):
    fg = torch.tensor([float(f0), float(g0)], dtype=torch.float32, requires_grad=True)
    f = fg[0]
    g = fg[1]

    O1 = torch.log(1.0 + 0.6 * f)
    O2 = torch.sqrt(f + 1.8 * g + 1e-6)
    O3 = torch.sqrt(f + 0.7 * g + 1e-6)
    outputs = [O1, O2, O3]

    J_rows = []
    for O in outputs:
        grad = torch.autograd.grad(O, fg, retain_graph=True)[0]
        J_rows.append(grad.detach().cpu().numpy())

    J = np.stack(J_rows, axis=0)
    _, S, _ = np.linalg.svd(J, full_matrices=False)
    sigma_max = float(S[0])
    sigma_min = float(S[-1])
    cond = float(sigma_max / sigma_min) if sigma_min > 0 else np.inf
    G = J.T @ J
    detG = float(np.linalg.det(G))
    return {
        'J': J,
        'sigma_max': sigma_max,
        'sigma_min': sigma_min,
        'cond': cond,
        'detG': detG,
    }


In [ ]:
# Evaluate conditioning at the midpoint of the learned radial profile for each sweep result
condition_rows = []
mid_idx = len(r) // 2

for row in results:
    f_mid = row['f_pred'][mid_idx].item()
    g_mid = row['g_pred'][mid_idx].item()
    metrics = jacobian_condition_metrics(f_mid, g_mid)
    condition_rows.append({
        'strength': row['strength'],
        'sigma_max': metrics['sigma_max'],
        'sigma_min': metrics['sigma_min'],
        'cond': metrics['cond'],
        'detG': metrics['detG'],
    })
    print(f"c={row['strength']:.2f} | sigma_min={metrics['sigma_min']:.6f} | cond={metrics['cond']:.6f} | detG={metrics['detG']:.6f}")


In [ ]:
# Extract a conditioning-based threshold when sigma_min falls below a heuristic level
sigma_min_vals = [row['sigma_min'] for row in condition_rows]
cond_vals = [row['cond'] for row in condition_rows]
detG_vals = [row['detG'] for row in condition_rows]

sigma_tol = max(sigma_min_vals) * 0.85
c_star_cond = None
for row in condition_rows:
    if c_star_cond is None and row['sigma_min'] < sigma_tol:
        c_star_cond = row['strength']
        break

print('sigma_tol:', sigma_tol)
print('conditioning-based c*:', c_star_cond)


In [ ]:
# Main figure: conditioning vs numerical threshold
strength_vals = [row['strength'] for row in results]
mean_err_vals = [row['abs_err_mean'] for row in results]
cond_strengths = [row['strength'] for row in condition_rows]

plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.plot(strength_vals, mean_err_vals, marker='o', label='mean abs error')
plt.axhline(mean_tolerance, linestyle='--', label='mean tolerance')
if c_star_num is not None:
    plt.axvline(c_star_num, linestyle=':', label=f'numeric c*={c_star_num:.2f}')
if c_star_cond is not None:
    plt.axvline(c_star_cond, linestyle='-.', label=f'conditioning c*={c_star_cond:.2f}')
plt.xlabel('corruption strength')
plt.ylabel('mean abs error')
plt.title('threshold comparison')
plt.legend()

plt.subplot(2, 2, 2)
plt.plot(cond_strengths, sigma_min_vals, marker='o', label='sigma_min')
plt.axhline(sigma_tol, linestyle='--', label='sigma_tol')
if c_star_cond is not None:
    plt.axvline(c_star_cond, linestyle=':', label=f'c*={c_star_cond:.2f}')
plt.xlabel('corruption strength')
plt.ylabel('smallest singular value')
plt.title('Jacobian collapse')
plt.legend()

plt.subplot(2, 2, 3)
plt.plot(cond_strengths, cond_vals, marker='o', color='tab:red', label='condition number')
plt.xlabel('corruption strength')
plt.ylabel('cond(J)')
plt.title('conditioning growth')
plt.legend()

plt.subplot(2, 2, 4)
plt.plot(cond_strengths, detG_vals, marker='o', color='tab:green', label='det(J^T J)')
plt.xlabel('corruption strength')
plt.ylabel('Gram determinant')
plt.title('area-based identifiability proxy')
plt.legend()

plt.suptitle('AdS4 Jacobian / Conditioning Sweep (v12)')
plt.tight_layout()
plt.savefig('ads4_phase_lock_v12_conditioning.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: ads4_phase_lock_v12_conditioning.png')


In [ ]:
# Reconstruction examples near low / numeric c* / high corruption
def nearest_index(target):
    diffs = [abs(row['strength'] - target) for row in results]
    return int(np.argmin(diffs))

pick_low = nearest_index(0.00)
pick_mid = nearest_index(c_star_num if c_star_num is not None else 0.16)
pick_high = nearest_index(0.30)

plt.figure(figsize=(12, 4))
for i, idx in enumerate([pick_low, pick_mid, pick_high], start=1):
    row = results[idx]
    plt.subplot(1, 3, i)
    plt.plot(r.cpu(), f_true(r).cpu(), label='f true')
    plt.plot(r.cpu(), row['f_pred'].cpu(), '--', label=f"f @{row['strength']:.2f}")
    plt.plot(r.cpu(), g_true(r).cpu(), label='g true')
    plt.plot(r.cpu(), row['g_pred'].cpu(), ':', label=f"g @{row['strength']:.2f}")
    plt.title(f"corruption={row['strength']:.2f}")
    plt.legend(fontsize=8)

plt.tight_layout()
plt.savefig('ads4_phase_lock_v12_reconstruction_examples.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: ads4_phase_lock_v12_reconstruction_examples.png')
